In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision


In [7]:
# Dataset and DataLoader

from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets


transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(), 
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    
])

trainset = datasets.ImageFolder("data/training_set", transform=transform)
testset = datasets.ImageFolder("data/test_set", transform=transform)



In [9]:
train_loader = DataLoader(trainset, batch_size=32, shuffle=True)
test_loader = DataLoader(testset, batch_size=32)


In [10]:
print(len(trainset))
print(len(testset))

8005
2023


In [22]:
# Build CNN

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(16*16*128, 512),
            nn.ReLU(),

            nn.Linear(512, 1)
        )

    def forward(self, x):

        x= self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x = self.fc_layers(x)
        x = torch.sigmoid(x)
        return x

        

In [23]:
model = CNN()

In [24]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [31]:
# Training the CNN

import torch
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        labels = labels.float().unsqueeze(1)

        output = model.forward(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoxh={epoch+1}/{epochs} & loss = {epoch_training_loss/len(train_loader)}")

epoxh=1/10 & loss = 0.6782391332535155
epoxh=2/10 & loss = 0.6044128830451889
epoxh=3/10 & loss = 0.5230124101220849
epoxh=4/10 & loss = 0.4748283473262749
epoxh=5/10 & loss = 0.41922983663728036
epoxh=6/10 & loss = 0.3364436226299559
epoxh=7/10 & loss = 0.2328230584285174
epoxh=8/10 & loss = 0.12735083208705086
epoxh=9/10 & loss = 0.05054345291545726
epoxh=10/10 & loss = 0.024743898030077347


In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        #images = images.to(device)
        outputs = model(images)

        preds = (outputs > 0.5).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

Accuracy: 0.749876421156698
Precision: 0.7346938775510204
Recall: 0.782608695652174
Confusion Matrix:
 [[725 286]
 [220 792]]
